In [1]:
import argparse
import os
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import torchvision.transforms.functional as RF
from PIL import Image
import numpy as np
import random,cv2,pandas,os,numpy
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from sklearn.preprocessing import OneHotEncoder

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

nz = 100
beta1 = 0.5
lr = 0.01
batch_size=47021
ngpu=1
ngf,nc = 3,3
ndf = 64
shape=(47021,10)

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

Random Seed:  999


In [4]:
train = pandas.read_csv('/kaggle/input/rohlik-sales-forecasting-challenge-v2/sales_train.csv', parse_dates=['date'])
test = pandas.read_csv('/kaggle/input/rohlik-sales-forecasting-challenge-v2/sales_test.csv', parse_dates=['date'])
ss = pandas.read_csv('/kaggle/input/rohlik-sales-forecasting-challenge-v2/solution.csv')

inventory = pandas.read_csv('/kaggle/input/rohlik-sales-forecasting-challenge-v2/inventory.csv')
weights = pandas.read_csv('/kaggle/input/rohlik-sales-forecasting-challenge-v2/test_weights.csv')
calendar = pandas.read_csv('/kaggle/input/rohlik-sales-forecasting-challenge-v2/calendar.csv', parse_dates=['date'])

Frankfurt_1 = calendar.query('date >= "2020-08-01 00:00:00" and warehouse =="Frankfurt_1"')
Prague_2 = calendar.query('date >= "2020-08-01 00:00:00" and warehouse =="Prague_2"')
Brno_1 = calendar.query('date >= "2020-08-01 00:00:00" and warehouse =="Brno_1"')
Munich_1 = calendar.query('date >= "2020-08-01 00:00:00" and warehouse =="Munich_1"')
Prague_3 = calendar.query('date >= "2020-08-01 00:00:00" and warehouse =="Prague_3"')
Prague_1 = calendar.query('date >= "2020-08-01 00:00:00" and warehouse =="Prague_1"')
Budapest_1 = calendar.query('date >= "2020-08-01 00:00:00" and warehouse =="Budapest_1"')

def process_calendar(df):
    """
    Обрабатывает календарный датафрейм, добавляя новые колонки:
    - days_to_holiday
    - days_to_shops_closed
    - day_after_closing
    - long_weekend
    - weekday
    """
    # Убеждаемся, что даты отсортированы
    df = df.sort_values('date').reset_index(drop=True)
    
    # 1. days_to_holiday
    df['next_holiday_date'] = df.loc[df['holiday'] == 1, 'date'].shift(-1)
    df['next_holiday_date'] = df['next_holiday_date'].bfill()
    df['days_to_holiday'] = (df['next_holiday_date'] - df['date']).dt.days
    df.drop(columns=['next_holiday_date'], inplace=True)
    
    # 2. days_to_shops_closed
    df['next_shops_closed_date'] = df.loc[df['shops_closed'] == 1, 'date'].shift(-1)
    df['next_shops_closed_date'] = df['next_shops_closed_date'].bfill()
    df['days_to_shops_closed'] = (df['next_shops_closed_date'] - df['date']).dt.days
    df.drop(columns=['next_shops_closed_date'], inplace=True)
    
    # 3. day_after_closing
    df['day_after_closing'] = (
        (df['shops_closed'] == 0) & (df['shops_closed'].shift(1) == 1)
    ).astype(int)
    
    # 4. long_weekend
    df['long_weekend'] = (
        (df['shops_closed'] == 1) & (df['shops_closed'].shift(1) == 1)
    ).astype(int)
    
    # 5. weekday
    df['weekday'] = df['date'].dt.weekday  # 0 (понедельник) - 6 (воскресенье)
    
    return df


# Список датафреймов
dfs = ['Frankfurt_1', 'Prague_2', 'Brno_1', 'Munich_1', 'Prague_3', 'Prague_1', 'Budapest_1']

# Применяем функцию ко всем датафреймам и собираем их в список
processed_dfs = [process_calendar(globals()[df]) for df in dfs]

# Конкатенируем все датафреймы в один
calendar_extended = pandas.concat(processed_dfs).sort_values('date').reset_index(drop=True)


test_calendar = test.merge(calendar_extended, on=['date', 'warehouse'], how='left')
test_data = test_calendar.merge(inventory, on=['unique_id', 'warehouse'], how='left').sort_values(['unique_id', 'date'])

test_data['YEAR'] = test_data['date'].dt.year
test_data['MONTH'] = test_data['date'].dt.month
test_data['DAY'] = test_data['date'].dt.day
test_data = test_data.drop(columns=['date'])

x = test_data

In [5]:
x = x[['unique_id', 'YEAR', 'MONTH', 'DAY', 'total_orders', 'sell_price_main',
        'type_0_discount', 'type_1_discount', 'type_2_discount',
        'type_3_discount', 'type_4_discount', 'type_5_discount',
        'type_6_discount', 'holiday', 'shops_closed',
        'winter_school_holidays', 'school_holidays', 'days_to_holiday',
        'days_to_shops_closed', 'day_after_closing', 'long_weekend', 'weekday',
        'product_unique_id']]

x = torch.from_numpy(x.to_numpy()).type('torch.FloatTensor')
x.shape

torch.Size([47021, 23])

In [6]:
y=pandas.read_csv("/kaggle/input/rohlik-sales-forecasting-challenge-v2/sales_test.csv").sort_values(['unique_id', 'date'])
y=y[['unique_id', 'date']].to_numpy()

In [7]:
test_x = torch.utils.data.DataLoader(x,batch_size=batch_size)
test_x_ = torch.utils.data.DataLoader(y,batch_size=batch_size)

In [8]:
def weights_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight.data, gain=0.02)
        
class RF_NET(nn.Module):
    def __init__(self):
        super().__init__()
        self.rafire = nn.Sequential(
            
            nn.Linear(23, 1524),
            nn.BatchNorm1d(1524),
            nn.ReLU(),
            
            nn.Linear(1524, 824),
            nn.BatchNorm1d(824),
            nn.ReLU(),
            
            nn.Linear(824, 424),
            nn.BatchNorm1d(424),
            nn.ReLU(),
            
            nn.Linear(424, 212),
            nn.BatchNorm1d(212),
            nn.ReLU(),
            
            nn.Linear(212, 124),
            nn.BatchNorm1d(124),
            nn.ReLU(),
            
            nn.Linear(124, 1)
        )

    def forward(self,x):
        #x,_=self.rafire_0(x)
        #print(x,x.shape)
        return self.rafire(x)

In [9]:
EFF_NET = RF_NET().float().eval()
EFF_NET= nn.DataParallel(EFF_NET).to(device)
EFF_NET.load_state_dict(torch.load("/kaggle/input/sales-forecasting/23.10028076171875.pth",weights_only=False,map_location=torch.device('cpu')))

<All keys matched successfully>

In [10]:
j=0
submission={'id' : [], 'sales_hat' : []}
for i in test_x:
    out=EFF_NET(i.to(device)).view(-1).detach().numpy()
for data in out:
    submission['id'].append(f"{str(y[j][0])}_{y[j][1]}")
    output = data
    submission['sales_hat'].append(output)
    j+=1
pandas.DataFrame(submission).to_csv(f"submission.csv", index=False)